QUESTION NUMBER 8, PAGE 286, CHAPTER 6 OF THE TEXTBOOK "AN INTRODUCTION TO STATISTICAL LEARNING WITH APPLICATIONS IN PYTHON".

In this exercise, we will generate simulated data, and will then use this data to perform forward and backward stepwise selection.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots
from statsmodels.api import OLS
import sklearn.model_selection as skm
import sklearn.linear_model as skl
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from ISLP import load_data
from ISLP.models import ModelSpec as MS
from functools import partial

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from ISLP.models import \
(Stepwise ,
sklearn_selected ,
sklearn_selection_path)
#pip install l0bnb
from l0bnb import fit_path

from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.feature_selection import SequentialFeatureSelector

In [2]:
# Reproducibility
rng = np.random.default_rng(seed=123)

def make_poly_df(X, degree=10):
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly = poly.fit_transform(X.reshape(-1, 1))
    cols = [f"X^{i}" for i in range(1, degree + 1)]
    return pd.DataFrame(X_poly, columns=cols)

def fit_full_sigma2(X_df, Y):
    # Estimate sigma^2 from full linear model with all predictors
    lr = LinearRegression().fit(X_df, Y)
    Yhat = lr.predict(X_df)
    rss = np.sum((Y - Yhat)**2)
    n = len(Y)
    p = X_df.shape[1]
    # Unbiased error variance estimate for linear model:
    # sigma^2_hat = RSS / (n - p - 1) if intercept is present.
    # Our LinearRegression has intercept by default, so use that.
    return rss / (n - p - 1)

def neg_cp_scorer_factory(sigma2):
    # Returns a scorer callable compatible with scikit-learn's SFS
    # SFS passes a fitted estimator and data for the held-out fold
    def scorer(estimator, X, y):
        yhat = estimator.predict(X)
        rss = np.sum((y - yhat)**2)
        n, p = X.shape
        # Mallows Cp (up to scaling): RSS + 2 p sigma^2; smaller is better.
        # We return negative so higher is better for scorer.
        return -(rss + 2 * p * sigma2) / n
    return scorer

def report_selected_and_coeffs(selector, model, X_df, Y):
    mask = selector.get_support()
    selected_cols = list(X_df.columns[mask])
    # Fit final model on selected predictors
    X_sel = X_df[selected_cols] if selected_cols else np.zeros((len(Y), 0))
    model.fit(X_sel, Y)
    # Coefficients: intercept and per-selected feature
    intercept = model.intercept_
    coefs = dict(zip(selected_cols, model.coef_)) if selected_cols else {}
    return selected_cols, intercept, coefs

QUESTION A

Create a random number generator and use its normal() method to generate a predictor X of length n = 100, as well as a noise vector ϵ of length n = 100.

In [3]:
n = 100
X = rng.normal(loc=0.0, scale=1.0, size=n)
epsilon = rng.normal(loc=0.0, scale=1.0, size=n)
X_df = make_poly_df(X, degree=10)
X_df.head()


,X^1,X^2,X^3,X^4,X^5,X^6,X^7,X^8,X^9,X^10
0,-0.989121,0.978361,-0.967718,0.957190,-0.946777,0.936478,-0.926290,0.916213,-9.062462e-01,8.963874e-01
1,-0.367787,0.135267,-0.049749,0.018297,-0.006729,0.002475,-0.000910,0.000335,-1.231299e-04,4.528555e-05
2,1.287925,1.658751,2.136348,2.751456,3.543670,4.563982,5.878068,7.570513,9.750255e+00,1.255760e+01
3,0.193974,0.037626,0.007298,0.001416,0.000275,0.000053,0.000010,0.000002,3.887766e-07,7.541272e-08
4,0.920231,0.846825,0.779274,0.717112,0.659909,0.607269,0.558827,0.514250,4.732290e-01,4.354799e-01


QUESTION B

Generate a response vector Y of length n = 100 according to the model

Y = β0 + β1X + β2X2 + β3X3 + ϵ,

where β0, β1, β2, and β3 are constants of your choice.

In [4]:
beta0, beta1, beta2, beta3 = 2.0, 1.5, -0.8, 0.5
Y = beta0 + beta1*X + beta2*(X**2) + beta3*(X**3) + epsilon


QUESTION C

Use forward stepwise selection in order to select a model containing the predictors X, X2, . . . , X10. What is the model obtained according to Cp? Report the coefficients of the model obtained.

In [5]:
# Estimate sigma^2 from the full model including all polynomial terms
sigma2_full = fit_full_sigma2(X_df, Y)
cp_scorer = neg_cp_scorer_factory(sigma2_full)

# Forward stepwise (greedy) with Cp
lr = LinearRegression()
sfs_forward_cp = SequentialFeatureSelector(
    lr,
    n_features_to_select="auto",     # let SFS pick best size
    direction="forward",
    scoring=cp_scorer,
    cv=5,                            # CV stabilizes selection
    n_jobs=-1
).fit(X_df, Y)

selected_fwd_cp, intercept_fwd_cp, coefs_fwd_cp = report_selected_and_coeffs(
    sfs_forward_cp, LinearRegression(), X_df, Y
)

print("Forward stepwise (Cp) selected predictors:", selected_fwd_cp)
print("Forward stepwise (Cp) intercept:", intercept_fwd_cp)
print("Forward stepwise (Cp) coefficients:")
for k, v in coefs_fwd_cp.items():
    print(f"  {k}: {v:.6f}")


Forward stepwise (Cp) selected predictors: ['X^1', 'X^2', 'X^3', 'X^5', 'X^7']
Forward stepwise (Cp) intercept: 2.0584300922083676
Forward stepwise (Cp) coefficients:
  X^1: 2.105866
  X^2: -0.712960
  X^3: -0.543520
  X^5: 0.472738
  X^7: -0.057847


In [6]:
sfs_forward_mse = SequentialFeatureSelector(
    LinearRegression(),
    n_features_to_select="auto",
    direction="forward",
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1
).fit(X_df, Y)

selected_fwd_mse, intercept_fwd_mse, coefs_fwd_mse = report_selected_and_coeffs(
    sfs_forward_mse, LinearRegression(), X_df, Y
)

print("\nForward stepwise (MSE) selected predictors:", selected_fwd_mse)
print("Forward stepwise (MSE) intercept:", intercept_fwd_mse)
print("Forward stepwise (MSE) coefficients:")
for k, v in coefs_fwd_mse.items():
    print(f"  {k}: {v:.6f}")



Forward stepwise (MSE) selected predictors: ['X^1', 'X^2', 'X^3', 'X^5', 'X^7']
Forward stepwise (MSE) intercept: 2.0584300922083676
Forward stepwise (MSE) coefficients:
  X^1: 2.105866
  X^2: -0.712960
  X^3: -0.543520
  X^5: 0.472738
  X^7: -0.057847


QUESTION D

Repeat (c), using backwards stepwise selection. How does your answer compare to the results in (c)?

In [7]:
sfs_backward_cp = SequentialFeatureSelector(
    LinearRegression(),
    n_features_to_select="auto",
    direction="backward",
    scoring=cp_scorer,
    cv=5,
    n_jobs=-1
).fit(X_df, Y)

selected_bwd_cp, intercept_bwd_cp, coefs_bwd_cp = report_selected_and_coeffs(
    sfs_backward_cp, LinearRegression(), X_df, Y
)

print("\nBackward stepwise (Cp) selected predictors:", selected_bwd_cp)
print("Backward stepwise (Cp) intercept:", intercept_bwd_cp)
print("Backward stepwise (Cp) coefficients:")
for k, v in coefs_bwd_cp.items():
    print(f"  {k}: {v:.6f}")

# Optional MSE comparison for backward
sfs_backward_mse = SequentialFeatureSelector(
    LinearRegression(),
    n_features_to_select="auto",
    direction="backward",
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1
).fit(X_df, Y)

selected_bwd_mse, intercept_bwd_mse, coefs_bwd_mse = report_selected_and_coeffs(
    sfs_backward_mse, LinearRegression(), X_df, Y
)

print("\nBackward stepwise (MSE) selected predictors:", selected_bwd_mse)
print("Backward stepwise (MSE) intercept:", intercept_bwd_mse)
print("Backward stepwise (MSE) coefficients:")
for k, v in coefs_bwd_mse.items():
    print(f"  {k}: {v:.6f}")

# Quick textual comparison
print("\nComparison: forward Cp vs backward Cp")
print("Forward Cp:", selected_fwd_cp)
print("Backward Cp:", selected_bwd_cp)



Backward stepwise (Cp) selected predictors: ['X^1', 'X^5', 'X^6', 'X^8', 'X^10']
Backward stepwise (Cp) intercept: 1.935015340232101
Backward stepwise (Cp) coefficients:
  X^1: 1.914029
  X^5: 0.129036
  X^6: -0.818665
  X^8: 0.402501
  X^10: -0.048697

Backward stepwise (MSE) selected predictors: ['X^1', 'X^5', 'X^6', 'X^8', 'X^10']
Backward stepwise (MSE) intercept: 1.935015340232101
Backward stepwise (MSE) coefficients:
  X^1: 1.914029
  X^5: 0.129036
  X^6: -0.818665
  X^8: 0.402501
  X^10: -0.048697

Comparison: forward Cp vs backward Cp
Forward Cp: ['X^1', 'X^2', 'X^3', 'X^5', 'X^7']
Backward Cp: ['X^1', 'X^5', 'X^6', 'X^8', 'X^10']


QUESTION E

Now fit a lasso model to the simulated data, again using X, X2, . . . , X10 as predictors. Use cross-validation to select the optimal value of λ. Create plots of the cross-validation error as a function of λ. Report the resulting coefficient estimates, and discuss the results obtained.

In [8]:
# LassoCV with scaling and polynomial features in a pipeline
lasso_pipe = make_pipeline(
    StandardScaler(with_mean=True),
    PolynomialFeatures(degree=10, include_bias=False),
    LassoCV(cv=10, random_state=42, max_iter=10000)
)
lasso_pipe.fit(X.reshape(-1, 1), Y)

lasso = lasso_pipe.named_steps['lassocv']
alphas = lasso.alphas_
mse_path = lasso.mse_path_  # shape (n_alphas, n_folds) in recent sklearn

# Plot mean CV MSE vs alpha (lambda)
mean_mse = mse_path.mean(axis=1)

plt.figure(figsize=(7, 4))
plt.plot(alphas, mean_mse, marker='o', lw=1)
plt.xscale('log')
plt.xlabel('Lambda (alpha)')
plt.ylabel('Mean CV MSE')
plt.title('LassoCV: Cross-validation error vs lambda')
plt.grid(True, alpha=0.3)
plt.show()

# Report coefficient estimates on the 1..10 powers after scaling
# We need to refit a simple pipeline to extract coefficients against the polynomial columns:
poly_only = PolynomialFeatures(degree=10, include_bias=False)
X_poly = poly_only.fit_transform(X.reshape(-1, 1))
scaler = StandardScaler(with_mean=True).fit(X_poly)
X_poly_scaled = scaler.transform(X_poly)

lasso_refit = LassoCV(cv=10, random_state=42, max_iter=10000).fit(X_poly_scaled, Y)
cols = [f"X^{i}" for i in range(1, 11)]
coef_series = pd.Series(lasso_refit.coef_, index=cols)

print("\nLasso selected coefficients (scaled features):")
print(coef_series.round(6))

print("\nLasso intercept:", lasso_refit.intercept_)
print("Chosen lambda (alpha):", lasso_refit.alpha_)


NameError: name 'LassoCV' is not defined

In [ ]:
# Regenerate Y with only the X^7 signal
beta0_f = 1.0
beta7_f = 3.0
epsilon_f = rng.normal(loc=0.0, scale=1.0, size=n)
Y_f = beta0_f + beta7_f * (X**7) + epsilon_f

X_df_f = make_poly_df(X, degree=10)
sigma2_full_f = fit_full_sigma2(X_df_f, Y_f)
cp_scorer_f = neg_cp_scorer_factory(sigma2_full_f)

# Forward stepwise (Cp) for the new response
sfs_forward_cp_f = SequentialFeatureSelector(
    LinearRegression(),
    n_features_to_select="auto",
    direction="forward",
    scoring=cp_scorer_f,
    cv=5,
    n_jobs=-1
).fit(X_df_f, Y_f)

selected_fwd_cp_f, intercept_fwd_cp_f, coefs_fwd_cp_f = report_selected_and_coeffs(
    sfs_forward_cp_f, LinearRegression(), X_df_f, Y_f
)

print("\nForward stepwise (Cp), Y = β0 + β7 X^7 + ε")
print("Selected predictors:", selected_fwd_cp_f)
print("Intercept:", intercept_fwd_cp_f)
print("Coefficients:")
for k, v in coefs_fwd_cp_f.items():
    print(f"  {k}: {v:.6f}")

# Lasso on the new response
lasso_pipe_f = make_pipeline(
    StandardScaler(with_mean=True),
    PolynomialFeatures(degree=10, include_bias=False),
    LassoCV(cv=10, random_state=42, max_iter=10000)
)
lasso_pipe_f.fit(X.reshape(-1, 1), Y_f)
lasso_f = lasso_pipe_f.named_steps['lassocv']

# Extract which powers have nonzero coefficients (on scaled features)
poly_only_f = PolynomialFeatures(degree=10, include_bias=False)
X_poly_f = poly_only_f.fit_transform(X.reshape(-1, 1))
scaler_f = StandardScaler(with_mean=True).fit(X_poly_f)
X_poly_scaled_f = scaler_f.transform(X_poly_f)
lasso_refit_f = LassoCV(cv=10, random_state=42, max_iter=10000).fit(X_poly_scaled_f, Y_f)

coef_series_f = pd.Series(lasso_refit_f.coef_, index=[f"X^{i}" for i in range(1, 11)])

print("\nLasso (CV) for Y = β0 + β7 X^7 + ε")
print(coef_series_f.round(6))
print("Intercept:", lasso_refit_f.intercept_)
print("Chosen lambda (alpha):", lasso_refit_f.alpha_)


Interpretation: Since the model was built solely on X^7, thus the X^7 has the highest coefficient for both forward stepwise and lasso, indicating the biggest influence to the model. It can also be said that for forward stepwise, it has milder regularization compared to Lasso, since it has 5 features worth noting, while lasso has only 3 predictors worth noting. Some other stuff found that may be worth noting:

1. Both methods perform well in identifying the true signal x^7.
2. Lasso is more parsimonious and regularized, but still influenced by multicollinearity.
3. Forward stepwise is more flexible but prone to including noise-driven terms.
4. In practice, Lasso is preferred for high-dimensional or correlated predictors, while stepwise can be useful for interpretability and small models.